# Notebook 2 — TBox Bootstrapping

## Objective

Construct the **frozen semantic vocabulary** used by the subsequent
Semantic Onboarding activities from a selected reference candidate semantic
model.

This notebook implements **TBox Bootstrapping**. The input is one
artifact-specific candidate semantic model generated by Notebook 1 and
selected as the reference model for the organizational corpus.

Unlike Notebook 1, this stage is **deterministic**: no LLM is used. The
reference Prolog model is parsed to identify predicate signatures, metadata
keys, and fact categories. These elements are consolidated into a controlled
TBox that constrains subsequent ABox generation.

## Inputs

- Selected reference candidate semantic model in Prolog.

## Outputs

- TBox represented in Prolog.
- Frozen controlled vocabulary containing:
  - predicate signatures;
  - allowed contract metadata keys; and
  - allowed clause fact types.

## Main Activities

1. Predicate extraction.
2. Metadata-key identification.
3. Fact-category identification.
4. Controlled vocabulary construction and freezing.

## Relation to the Methodology

The implementation progression is:

`Selected Candidate Semantic Model → TBox Bootstrapping → Frozen TBox`

The TBox produced here is an **implementation-level semantic vocabulary**.
It is distinct from the foundational/service-contract ontology used later in
Notebook 4 for Ontology Projection.

## 1. Environment Setup

The reference implementation is designed to run in Google Colab and uses
Google Drive for persistence. This notebook does not invoke an LLM or any
external generative service.

In [ ]:
# Mount Google Drive to access the selected reference model and save the TBox.
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
# Import required standard-library modules.
from collections import Counter, defaultdict
from pathlib import Path
import re

## 2. Configuration

`REFERENCE_MODEL_FILE` must point to the candidate semantic model selected
after Notebook 1. In the thesis implementation, the selected contract model
acts as the reference from which the controlled vocabulary is bootstrapped.

The output file contains only schema/vocabulary declarations. It does not
copy contract individuals or instance-level facts from the reference model.

In [ ]:
MODEL_DIRECTORY = Path(
    "/content/drive/My Drive/KOA/onboarding/modelos"
)

# Selected reference candidate semantic model generated by Notebook 1.
REFERENCE_MODEL_FILE = MODEL_DIRECTORY / "KOA_195_2022_Brasoftware.pl"

# Frozen TBox used to constrain Notebook 3 — ABox Generation.
TBOX_OUTPUT_FILE = MODEL_DIRECTORY / "KOA_tbox_bootstrap.pl"

print(f"Reference semantic model: {REFERENCE_MODEL_FILE}")
print(f"TBox output:             {TBOX_OUTPUT_FILE}")

## 3. Prolog Parsing Utilities

The following functions parse the selected reference model without executing
it. Their purpose is limited to identifying the structural vocabulary
actually present in the candidate model.

The parser extracts predicate names and arities from Prolog facts and rules,
while preserving nested terms and quoted strings when splitting arguments.

In [ ]:
def sanitize_prolog_text(text: str) -> str:
    """Remove control characters that can interfere with lightweight parsing."""
    return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", text)


def split_prolog_clauses(text: str) -> list[str]:
    """
    Split Prolog source into clauses terminated by a period outside quoted
    strings. This is a lightweight parser for the generated models used in
    the reference implementation.
    """
    clauses = []
    buffer = []
    in_string = False
    quote_char = None
    escaped = False

    for index, char in enumerate(text):
        buffer.append(char)

        if in_string:
            if escaped:
                escaped = False
                continue
            if char == "\\":
                escaped = True
                continue
            if char == quote_char:
                in_string = False
            continue

        if char in ("'", '"'):
            in_string = True
            quote_char = char
            continue

        if char == ".":
            next_index = index + 1
            if next_index >= len(text) or text[next_index].isspace():
                clause = "".join(buffer).strip()
                if clause:
                    clauses.append(clause)
                buffer = []

    remainder = "".join(buffer).strip()
    if remainder:
        clauses.append(remainder)

    return clauses


def split_arguments(arguments_source: str) -> list[str]:
    """Split a Prolog argument list while respecting nested terms and strings."""
    arguments = []
    buffer = []
    in_string = False
    quote_char = None
    escaped = False
    paren_depth = 0
    bracket_depth = 0

    for char in arguments_source:
        if in_string:
            buffer.append(char)
            if escaped:
                escaped = False
                continue
            if char == "\\":
                escaped = True
                continue
            if char == quote_char:
                in_string = False
            continue

        if char in ("'", '"'):
            in_string = True
            quote_char = char
            buffer.append(char)
            continue

        if char == "(":
            paren_depth += 1
        elif char == ")":
            paren_depth -= 1
        elif char == "[":
            bracket_depth += 1
        elif char == "]":
            bracket_depth -= 1

        if char == "," and paren_depth == 0 and bracket_depth == 0:
            arguments.append("".join(buffer).strip())
            buffer = []
        else:
            buffer.append(char)

    remainder = "".join(buffer).strip()
    if remainder:
        arguments.append(remainder)

    return arguments


def parse_clause_head(clause: str):
    """
    Return (predicate_name, arguments) for the head of a Prolog clause.
    Comments, directives, and malformed clauses are ignored.
    """
    clause = clause.strip()

    if (
        not clause
        or clause.startswith("%")
        or clause.startswith("```")
        or clause.startswith(":-")
    ):
        return None

    head = clause.split(":-", 1)[0].strip()

    if head.endswith("."):
        head = head[:-1].strip()

    match = re.match(
        r"^([a-z_][A-Za-z0-9_]*)\\s*\\((.*)\\)$",
        head,
        flags=re.DOTALL,
    )
    if not match:
        return None

    predicate_name = match.group(1)
    arguments_source = match.group(2).strip()
    arguments = (
        split_arguments(arguments_source)
        if arguments_source
        else []
    )

    return predicate_name, arguments


def collect_predicate_signatures(text: str):
    """
    Collect predicate signatures and a small number of examples for
    inspection.
    """
    counts = Counter()
    examples = defaultdict(list)

    for clause in split_prolog_clauses(text):
        parsed = parse_clause_head(clause)
        if not parsed:
            continue

        predicate_name, arguments = parsed
        signature = (predicate_name, len(arguments))
        counts[signature] += 1

        if len(examples[signature]) < 3:
            examples[signature].append(arguments)

    return counts, examples

## 4. Controlled Vocabulary Extraction

TBox Bootstrapping freezes two vocabulary dimensions that are subsequently
enforced during ABox Generation:

- metadata keys appearing as the second argument of
  `contract_metadata/3`; and
- fact types appearing as the third argument of
  `contract_clause_fact/5`.

Predicate signatures are also preserved so that the TBox records the
structural forms admitted by the selected reference semantic model.

Values that cannot be mapped to this controlled vocabulary in Notebook 3
must not silently extend the TBox; they are handled separately as unmapped
elements.

In [ ]:
def normalize_atom(atom: str) -> str:
    """Normalize a simple unquoted Prolog atom used as vocabulary."""
    return atom.strip().lower()


def collect_allowed_metadata_keys(text: str) -> list[str]:
    """
    Extract metadata keys from:
    contract_metadata(ContractId, Key, Value).
    """
    keys = set()

    for clause in split_prolog_clauses(text):
        parsed = parse_clause_head(clause)
        if not parsed:
            continue

        predicate_name, arguments = parsed
        if predicate_name == "contract_metadata" and len(arguments) == 3:
            key = arguments[1].strip()
            if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key):
                keys.add(normalize_atom(key))

    return sorted(keys)


def collect_allowed_fact_types(text: str) -> list[str]:
    """
    Extract fact categories from:
    contract_clause_fact(ContractId, ClauseId, FactType, Data, Evidence).
    """
    fact_types = set()

    for clause in split_prolog_clauses(text):
        parsed = parse_clause_head(clause)
        if not parsed:
            continue

        predicate_name, arguments = parsed
        if predicate_name == "contract_clause_fact" and len(arguments) == 5:
            fact_type = arguments[2].strip()
            if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", fact_type):
                fact_types.add(normalize_atom(fact_type))

    return sorted(fact_types)

## 5. TBox Construction

The generated TBox deliberately avoids treating predicate arity as an
ontological classification such as object property or datatype property.
At this stage, the goal is to freeze the **controlled semantic vocabulary**,
not to perform foundational ontology grounding.

Ontological commitments are introduced later, during Notebook 4 —
Ontology Projection.

In [ ]:
def generate_tbox(
    predicate_signatures,
    metadata_keys: list[str],
    fact_types: list[str],
    reference_model_name: str,
) -> str:
    """Generate the frozen implementation-level TBox in Prolog."""
    lines = [
        "% =====================================================================",
        "% KOA Frozen TBox",
        "% Generated deterministically from a selected candidate semantic model.",
        "% This file contains vocabulary/schema declarations only.",
        "% It contains no contract individuals or ABox assertions.",
        "% =====================================================================",
        "",
        f"tbox_reference_model('{reference_model_name}').",
        "",
        "% --- Predicate signatures --------------------------------------------",
    ]

    for predicate_name, arity in sorted(predicate_signatures):
        lines.append(
            f"allowed_predicate_signature({predicate_name}, {arity})."
        )

    lines.extend([
        "",
        "% --- Allowed contract metadata keys ----------------------------------",
    ])

    for key in metadata_keys:
        lines.append(f"allowed_metadata_key({key}).")

    lines.extend([
        "",
        "% --- Allowed clause fact types ---------------------------------------",
    ])

    for fact_type in fact_types:
        lines.append(f"allowed_fact_type({fact_type}).")

    lines.extend([
        "",
        "% --- Unmapped-element policy -----------------------------------------",
        "% Values that do not fit the frozen vocabulary must not extend the TBox",
        "% automatically. Notebook 3 records such elements separately so they can",
        "% be inspected without changing the controlled vocabulary.",
        "",
    ])

    return "\\n".join(lines)

## 6. Execute TBox Bootstrapping

The selected reference semantic model is read, inspected, and transformed
into the frozen TBox. The execution summary reports the vocabulary elements
that were identified before the resulting Prolog file is persisted.

In [ ]:
if not REFERENCE_MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Reference semantic model not found: {REFERENCE_MODEL_FILE}"
    )

MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

reference_text = REFERENCE_MODEL_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)
reference_text = sanitize_prolog_text(reference_text)

predicate_counts, predicate_examples = collect_predicate_signatures(
    reference_text
)
metadata_keys = collect_allowed_metadata_keys(reference_text)
fact_types = collect_allowed_fact_types(reference_text)

print(f"Predicate signatures: {len(predicate_counts)}")
print(f"Metadata keys:        {len(metadata_keys)}")
print(f"Clause fact types:    {len(fact_types)}")

In [ ]:
print("Predicate signatures identified:")
for (predicate_name, arity), count in sorted(predicate_counts.items()):
    print(f" - {predicate_name}/{arity}: {count} occurrence(s)")

print("\nAllowed metadata keys:")
for key in metadata_keys:
    print(f" - {key}")

print("\nAllowed clause fact types:")
for fact_type in fact_types:
    print(f" - {fact_type}")

In [ ]:
tbox_content = generate_tbox(
    predicate_signatures=predicate_counts.keys(),
    metadata_keys=metadata_keys,
    fact_types=fact_types,
    reference_model_name=REFERENCE_MODEL_FILE.name,
)

TBOX_OUTPUT_FILE.write_text(tbox_content, encoding="utf-8")

print(f"Frozen TBox saved to: {TBOX_OUTPUT_FILE}")

## 7. Output and Next Step

The generated Prolog file is the **frozen TBox** used by the next
Semantic Onboarding activity.

The TBox records:

- predicate signatures admitted by the selected reference semantic model;
- controlled contract metadata keys; and
- controlled clause fact types.

It does **not** contain instance-level contract knowledge and it does
**not** perform UFO/service-contract ontology grounding.

The next implementation stage is:

**Notebook 3 — ABox Generation**

Each organizational artifact is transformed into an artifact-specific ABox
under this frozen vocabulary. Elements that cannot be mapped to the TBox are
retained separately rather than silently extending the controlled
vocabulary.

The implementation progression is:

`Candidate Semantic Models → Frozen TBox → Controlled ABox Generation`